
## Folder 04 — File 04: Silver Order Items:


**Transformation rules:**

- Trim order_item_id, order_id, and product_id.
- Convert quantity to INTEGER.
- Convert unit_price and line_total to DECIMAL(12,2).
- Convert discount_rate to DECIMAL(5,4).
- Reject missing identifiers, invalid numeric values, zero or negative quantities, negative prices, and discounts outside the range 0–1.
- Deduplicate using order_item_id.
- Save valid and invalid records separately.

Business rule: **A discount rate of 0.15 means 15%. It is not 15**.

In [0]:

## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_order_items"
)

SILVER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_order_items"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_order_items"
)


## 3. Create Silver schema

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)

print("Silver schema is ready.")


## 4. Read Bronze order items

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise Exception(
        f"Bronze table not found: {BRONZE_TABLE}. "
        "Run Folder 03 first."
    )

bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze order item records: {bronze_df.count()}")

display(bronze_df.limit(10))


 ## 5. Clean and standardize order item columns

cleaned_df = (
    bronze_df

    .withColumn(
        "order_item_id",
        F.trim(F.col("order_item_id").cast("string"))
    )
    .withColumn(
        "order_id",
        F.trim(F.col("order_id").cast("string"))
    )
    .withColumn(
        "product_id",
        F.trim(F.col("product_id").cast("string"))
    )

    # Convert quantity into INTEGER

    .withColumn(
        "quantity",
        F.expr("try_cast(quantity AS INT)")
    )

    # Convert prices into DECIMAL(12,2)

    .withColumn(
        "unit_price",
        F.expr("try_cast(unit_price AS DECIMAL(12,2))")
    )
    .withColumn(
        "line_total",
        F.expr("try_cast(line_total AS DECIMAL(12,2))")
    )

    # Convert discount rate into DECIMAL(5,4)

    .withColumn(
        "discount_rate",
        F.expr("try_cast(discount_rate AS DECIMAL(5,4))")
    )
)

print("Order item standardization completed.")

display(cleaned_df.limit(10))


## 6. Identify invalid order item records

invalid_condition = (
    F.col("order_item_id").isNull()
    | (F.col("order_item_id") == "")

    | F.col("order_id").isNull()
    | (F.col("order_id") == "")

    | F.col("product_id").isNull()
    | (F.col("product_id") == "")

    | F.col("quantity").isNull()
    | (F.col("quantity") <= 0)

    | F.col("unit_price").isNull()
    | (F.col("unit_price") < 0)

    | F.col("discount_rate").isNull()
    | (F.col("discount_rate") < 0)
    | (F.col("discount_rate") > 1)

    | F.col("line_total").isNull()
    | (F.col("line_total") < 0)
)

invalid_df = (
    cleaned_df
    .filter(invalid_condition)
    .withColumn(
        "_quarantine_reason",
        F.concat_ws(
            "; ",

            F.when(
                F.col("order_item_id").isNull()
                | (F.col("order_item_id") == ""),
                F.lit("Missing order_item_id")
            ),

            F.when(
                F.col("order_id").isNull()
                | (F.col("order_id") == ""),
                F.lit("Missing order_id")
            ),

            F.when(
                F.col("product_id").isNull()
                | (F.col("product_id") == ""),
                F.lit("Missing product_id")
            ),

            F.when(
                F.col("quantity").isNull(),
                F.lit("Missing or invalid quantity")
            ),

            F.when(
                F.col("quantity") <= 0,
                F.lit("Quantity must be greater than zero")
            ),

            F.when(
                F.col("unit_price").isNull(),
                F.lit("Missing or invalid unit_price")
            ),

            F.when(
                F.col("unit_price") < 0,
                F.lit("Negative unit_price")
            ),

            F.when(
                F.col("discount_rate").isNull(),
                F.lit("Missing or invalid discount_rate")
            ),

            F.when(
                (F.col("discount_rate") < 0)
                | (F.col("discount_rate") > 1),
                F.lit("Discount rate must be between 0 and 1")
            ),

            F.when(
                F.col("line_total").isNull(),
                F.lit("Missing or invalid line_total")
            ),

            F.when(
                F.col("line_total") < 0,
                F.lit("Negative line_total")
            )
        )
    )
)

valid_df = cleaned_df.filter(~invalid_condition)

print(f"Valid order items: {valid_df.count()}")
print(f"Invalid order items: {invalid_df.count()}")


## 7. Remove duplicate order item records

deduplication_window = (
    Window
    .partitionBy("order_item_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc_nulls_last(),
        F.col("_source_file").desc_nulls_last()
    )
)

deduplicated_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(deduplication_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

print(
    f"Records after deduplication: "
    f"{deduplicated_df.count()}"
)


## 8. Save invalid records to quarantine

(
    invalid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Quarantine table created: {QUARANTINE_TABLE}")


## 9. Save clean records to Silver Delta table

(
    deduplicated_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table created: {SILVER_TABLE}")


## 10. Validate Silver order item records

silver_df = spark.table(SILVER_TABLE)

total_silver_records = silver_df.count()

duplicate_item_groups = (
    silver_df
    .groupBy("order_item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_silver_records = (
    silver_df
    .filter(
        F.col("order_item_id").isNull()
        | F.col("order_id").isNull()
        | F.col("product_id").isNull()
        | F.col("quantity").isNull()
        | (F.col("quantity") <= 0)
        | F.col("unit_price").isNull()
        | (F.col("unit_price") < 0)
        | F.col("discount_rate").isNull()
        | (F.col("discount_rate") < 0)
        | (F.col("discount_rate") > 1)
        | F.col("line_total").isNull()
        | (F.col("line_total") < 0)
    )
    .count()
)

print("=" * 50)
print("SILVER ORDER ITEM VALIDATION")
print("=" * 50)

print(f"Silver records: {total_silver_records}")
print(f"Duplicate item groups: {duplicate_item_groups}")
print(f"Invalid Silver records: {invalid_silver_records}")

if duplicate_item_groups > 0 or invalid_silver_records > 0:
    raise Exception("Silver order item validation failed.")

print("Silver order item validation passed.")


## 11. Calculate order item metrics

display(
    silver_df
    .agg(
        F.count("*").alias("total_order_items"),
        F.sum("quantity").alias("total_units_sold"),
        F.sum("line_total").alias("total_line_sales"),
        F.avg("discount_rate").alias("average_discount_rate")
    )
)


## 12. Display final Silver order item data

display(
    silver_df.orderBy("order_item_id").limit(20)
)


## 13. Final summary

print("=" * 50)
print("ORDER ITEM SILVER TRANSFORMATION COMPLETED")
print("=" * 50)

print(f"Bronze table     : {BRONZE_TABLE}")
print(f"Silver table     : {SILVER_TABLE}")
print(f"Quarantine table : {QUARANTINE_TABLE}")

print(f"Bronze records   : {bronze_df.count()}")
print(f"Silver records   : {total_silver_records}")
print(f"Invalid records  : {invalid_df.count()}")

print("=" * 50)
